In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error,r2_score

In [21]:
df=pd.read_csv("/home/backup/ml_datasets/pima_indians_diabetes.csv")
print(df.head(10))
print(df.info)
print(df.isnull().sum())
print(df.shape)

   preg   plas  pres  skin   insu  mass   pedi   age            class
0   6.0  148.0  72.0  35.0    0.0  33.6  0.627  50.0  tested_positive
1   1.0   85.0  66.0  29.0    0.0  26.6  0.351  31.0  tested_negative
2   8.0  183.0  64.0   0.0    0.0  23.3  0.672  32.0  tested_positive
3   1.0   89.0  66.0  23.0   94.0  28.1  0.167  21.0  tested_negative
4   0.0  137.0  40.0  35.0  168.0  43.1  2.288  33.0  tested_positive
5   5.0  116.0  74.0   0.0    0.0  25.6  0.201  30.0  tested_negative
6   3.0   78.0  50.0  32.0   88.0  31.0  0.248  26.0  tested_positive
7  10.0  115.0   0.0   0.0    0.0  35.3  0.134  29.0  tested_negative
8   2.0  197.0  70.0  45.0  543.0  30.5  0.158  53.0  tested_positive
9   8.0  125.0  96.0   0.0    0.0   0.0  0.232  54.0  tested_positive
<bound method DataFrame.info of      preg   plas  pres  skin   insu  mass   pedi   age            class
0     6.0  148.0  72.0  35.0    0.0  33.6  0.627  50.0  tested_positive
1     1.0   85.0  66.0  29.0    0.0  26.6  0.351  31.0

In [22]:
df['class']=df['class'].map({'tested_positive':1,'tested_negative':0})
x=df.drop('class',axis=1)
print(x)
y=df['class']
print(y)

     preg   plas  pres  skin   insu  mass   pedi   age
0     6.0  148.0  72.0  35.0    0.0  33.6  0.627  50.0
1     1.0   85.0  66.0  29.0    0.0  26.6  0.351  31.0
2     8.0  183.0  64.0   0.0    0.0  23.3  0.672  32.0
3     1.0   89.0  66.0  23.0   94.0  28.1  0.167  21.0
4     0.0  137.0  40.0  35.0  168.0  43.1  2.288  33.0
..    ...    ...   ...   ...    ...   ...    ...   ...
763  10.0  101.0  76.0  48.0  180.0  32.9  0.171  63.0
764   2.0  122.0  70.0  27.0    0.0  36.8  0.340  27.0
765   5.0  121.0  72.0  23.0  112.0  26.2  0.245  30.0
766   1.0  126.0  60.0   0.0    0.0  30.1  0.349  47.0
767   1.0   93.0  70.0  31.0    0.0  30.4  0.315  23.0

[768 rows x 8 columns]
0      1
1      0
2      1
3      0
4      1
      ..
763    0
764    0
765    0
766    1
767    0
Name: class, Length: 768, dtype: int64


In [23]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
print("Split done")

Split done


In [24]:
linear_model=LinearRegression()
linear_model.fit(x_train,y_train)
linear_pred=linear_model.predict(x_test)
lin_comp=pd.DataFrame({"Actual":y_test,"Predicted":linear_pred})
print(lin_comp)

     Actual  Predicted
668       0   0.335500
324       0   0.238099
624       0   0.151052
690       0   0.240136
473       0   0.481424
..      ...        ...
355       1   0.681590
534       0   0.163167
344       0   0.481562
296       1   0.301017
462       0   0.261109

[154 rows x 2 columns]


In [27]:
linear_mse=mean_squared_error(y_test,linear_pred)
linear_r2=r2_score(y_test,linear_pred)
print("****LINEAR REGRESSION******")
print("MSE:",linear_mse)
print("R2 Score:",linear_r2)

****LINEAR REGRESSION******
MSE: 0.171045272808501
R2 Score: 0.2550028117674177


In [29]:
#Ridge
ridge=Ridge()
ridge_param={"alpha":[0.1,1,10]}
ridge_cv=GridSearchCV(ridge,ridge_param,cv=5)
ridge_cv.fit(x_train,y_train)
ridge_model=ridge_cv.best_estimator_
ridge_pred=ridge_model.predict(x_test)
ridge_mse=mean_squared_error(y_test,ridge_pred)
ridge_r2=r2_score(y_test,ridge_pred)
print("******RIDGE REGRESSION********")
print("MSE:",ridge_mse)
print("R2 Score:",ridge_r2)

******RIDGE REGRESSION********
MSE: 0.17159157208036954
R2 Score: 0.2526233749388346


In [47]:
#LASSO
lasso=Lasso(max_iter=10000)
lasso_param={"alpha":[0.001,0.01,0.1]}
lasso_cv=GridSearchCV(lasso,lasso_param,cv=5)
lasso_cv.fit(x_train,y_train)
lasso_model=lasso_cv.best_estimator_
lasso_pred=lasso_model.predict(x_test)
lasso_mse=mean_squared_error(y_test,lasso_pred)
lasso_r2=r2_score(y_test,lasso_pred)
print("******LASSO ******")
print("MSE:",lasso_mse)
print("R2 Score:",lasso_r2)

******LASSO ******
MSE: 0.1714270479249324
R2 Score: 0.253339969038072


In [50]:
#results
print("Best alpha for Ridge:",ridge_cv.best_params_["alpha"])
print("Best alpha for Lasso:",lasso_cv.best_params_["alpha"])
results=pd.DataFrame({"Model":["Linear","Ridge","Lasso"],"MSE":[linear_mse,ridge_mse,lasso_mse],"R2 Score":[linear_r2,ridge_r2,lasso_r2]})
print(results)

Best alpha for Ridge: 10
Best alpha for Lasso: 0.001
    Model       MSE  R2 Score
0  Linear  0.171045  0.255003
1   Ridge  0.171592  0.252623
2   Lasso  0.171427  0.253340


In [53]:
print("\nModel Performance")
print("-"*55)
print(f"{'Model':<20}{'MSE':<15}{'R2 Score'}")
print("-"*55)
print(f"{'Linear Regression':<20}{linear_mse:<15.2f}{linear_r2:.4f}")
print(f"{'Ridge Regression':<20}{ridge_mse:<15.2f}{ridge_r2:.4f}")
print(f"{'Lasso Regression':<20}{lasso_mse:<15.2f}{lasso_r2:.4f}")


Model Performance
-------------------------------------------------------
Model               MSE            R2 Score
-------------------------------------------------------
Linear Regression   0.17           0.2550
Ridge Regression    0.17           0.2526
Lasso Regression    0.17           0.2533
